First, let's install the `strauss` code!

In [ ]:
 %pip install --quiet strauss

In [ ]:
%reload_ext autoreload 
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, Video, Audio, HTML as IPyHTML
import os
import numpy as np
import json

import ipywidgets as widgets
from ipywidgets import (
    Label, Layout, Box, VBox, HBox, GridBox, Button,
    IntSlider, FloatSlider, FloatLogSlider, ToggleButton,
    Accordion, Text, FloatText, IntText, BoundedFloatText,
    ToggleButtons, Checkbox, Output, HTML
)

from moviepy.editor import VideoFileClip, AudioFileClip

# strauss imports
from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss.score import Score
from strauss.generator import Synthesizer

We will use the `strauss` python library developed by **Dr James Trayford** to create the sonification of the data. - for more details on the library, please refer to the [documentation](https://strauss.readthedocs.io/en/latest/start.html)

In [ ]:
# Spain bounding rectangle
LAT_MIN = 36.0
LAT_MAX = 45.0

LON_MIN = -9.5
LON_MAX = 6.0

def load_data(file_path):
    with open(file_path, "r") as f:
        eclipse_data = json.load(f)

    # Extract the coordinates of the eclipse path from the JSON data
    coords = (eclipse_data["response"]["data"][0]
            ["visibilityLines"]["features"][0]
            ["geometry"]["coordinates"][0]
            )

    # Extract longitude and latitude from the coordinates, converting to float
    lons = np.array([float(c[0]) for c in coords], dtype=np.float64)
    lats = np.array([float(c[1]) for c in coords], dtype=np.float64)

    # Normalize latitudes and longitudes to [0, 1] range 
    lat_norm = (lats - LAT_MIN) / (LAT_MAX - LAT_MIN)
    lon_norm = (lons - LON_MIN) / (LON_MAX - LON_MIN)
    
    return lon_norm, lat_norm

In [ ]:
class Sonifier:
    def __init__(self, system, duration):
        self.system = system
        self.duration = duration 

    def generate(self, x, y, prop_x, prop_y):
        generator = Synthesizer()
        generator.load_preset('pitch_mapper')
        generator.modify_preset({'filter':'on'})
        notes = [["C2","D2","E2","G2","B2","C3","D3","E3","G3","B3"]]
        score =  Score(notes, self.duration)
    
        time = np.linspace(0, self.duration, len(x))

        data = {'pitch':list(range(10)),
                'time_evo':[time]*10,
                prop_x: [x]*10,
                prop_y: [y]*10
                }
        lims = {
            'time_evo': ('0%', '100%'), 
            'pitch_shift': ('0%', '100%'), 
            'cutoff': ('0%', '100%'),
            "volume": ('0%', '100%'), 
            "phi": (-0.5, 1.5),
        }

        # set up source
        sources = Objects(data.keys())
        sources.fromdict(data)
        sources.apply_mapping_functions(map_lims=lims)

        return Sonification(score, sources, generator, self.system)

In [ ]:
def make_video():
    video = VideoFileClip('data/eclipse_animation.mp4')
    audio = AudioFileClip('eclipse_sonification.wav')
    final = video.set_audio(audio)
    final.write_videofile('eclipse_sonification.mp4')

In [ ]:
CARD_STYLE = dict(
    border='1px solid #ccc',
    padding='10px',
    margin='5px',
    flex_flow='column',
    align_items='stretch',
    justify_content='flex-start',
    width='100%',
    overflow='visible'
)

CARD_TITLE_STYLE = dict(font_weight='bold', font_size='14px')
LABEL_STYLE = dict(font_weight='bold', font_size='11px')
MUTED_LABEL_STYLE = dict(font_size='12px', color='#666')


class SonificationUI:
    """
    Interactive UI for designing and generating sonifications of MAG data.
    
    This class provides a flexible, responsive interface for users to:
    - Choose which data variables to sonify
    - Map data to evolvable sound properties
    - Configure sonification duration
    - Generate and display sonifications with accompanying plots
    
    Attributes:
    -----------
    evolvable_properties : list
        Sound properties that can be mapped to data: pitch_shift, cutoff, volume, etc.
    """


    def __init__(self):
        """
        Initialize the SonificationUI.
        """

        self.evolvable_properties = [
            "pitch_shift", "cutoff", "volume", "phi"
        ]
        self.data_lon, self.data_lat = load_data("data/reducedtotaleclipsepath.json")
        self._create_widgets()
        self._setup_callbacks()


    def _create_widgets(self):
        """Create all UI components organized into cards."""
        # Video display area
        self.video_area = widgets.Output(layout=Layout(
            flex='1 1 auto',
            overflow='auto',
            border='1px solid #ccc',
            min_width='500px',
            min_height='400px'
        ))
        
        # Output area for sonification display
        self.output_area = widgets.Output(layout=Layout(
            flex='1 1 auto',
            overflow='auto',
            border='0',
            min_width='400px',
            min_height='200px'
        ))

        # Build individual cards
        self._create_data_card()
        self._create_settings_card()
        
        # Stack cards vertically with flexible sizing
        self.selector_panel = VBox([
            self.data_card,
            self.settings_card
        ], layout=Layout(
            flex='0 0 auto',
            overflow='visible',
            border='0'
        ))

        with self.video_area:
            # Display video
            video_file = "data/eclipse_animation.mp4"
            if os.path.exists(video_file):
                display(Video(video_file, width=500, height=400))
            else:
                print(f"⚠ Video file '{video_file}' not found")


    def _create_card(self, title, children, tag):
        """
        Helper method to create a styled card widget.
        
        Parameters:
        -----------
        title : str
            Card title text
        children : list
            List of child widgets to place in the card
        tag : str
            Identifier tag for finding the card later
            
        Returns:
        --------
        Box
            Styled card widget
        """
        title_widget = Label(value=title, style=CARD_TITLE_STYLE)
        
        card = Box(
            children=[title_widget] + children,
            layout=Layout(**CARD_STYLE)
        )
        card.tag = tag
        return card


    def _create_data_card(self):
        self.property_selector_1 = widgets.Dropdown(
            options=self.evolvable_properties, value='pitch_shift',
            layout=Layout(width='100%', overflow='visible')
        )
        self.property_selector_1.tag = 'property_selector_1'
        
        data1_section = VBox([
            Label(value='Longitude mapped to:', style=LABEL_STYLE),
            self.property_selector_1
        ], layout=Layout(width='100%', overflow='visible'))
        
        
        self.property_selector_2 = widgets.Dropdown(
            options=self.evolvable_properties, value='volume',
            layout=Layout(width='100%', overflow='visible')
        )
        self.property_selector_2.tag = 'property_selector_2'
        
        data2_section = VBox([
            Label(value='Latitude mapped to:', style=LABEL_STYLE),
            self.property_selector_2
        ], layout=Layout(width='100%', overflow='visible'))
        
        self.data_card = self._create_card(
            'Data & Mappings',
            [data1_section, data2_section],
            'data_card'
        )


    def _create_settings_card(self):
        """Create sonification settings card with duration and generate button."""
        self.length_selector = IntText(
            value=10,
            layout=Layout(width='100%')
        )
        self.length_selector.tag = 'length_selector'
        
        # length_section = VBox([
        #     Label(value='Duration (s):', style=LABEL_STYLE),
        #     self.length_selector
        # ], layout=Layout(width='100%', overflow='visible'))
        
        self.generate_button = Button(
            description='Generate', 
            button_style='success',
            tooltip='Generate sonification',
            icon='play',
            layout=Layout(width='100%')
        )
        self.generate_button.tag = 'generate_button'
        
        self.settings_card = self._create_card(
            'Settings',
            # [length_section, self.generate_button],
            [self.generate_button],
            'settings_card'
        )


    def _setup_callbacks(self):
        """Connect widget observers to their callback methods."""
        self.generate_button.on_click(self._on_generate_click)


    def _on_generate_click(self, button):
        """Handle Generate button click event - fetch and sonify."""
        with self.video_area:
            clear_output()
            try:
                data_1 = self.data_lon
                data_2 = self.data_lat
                prop_1 = self.property_selector_1.value
                prop_2 = self.property_selector_2.value
                duration = 15
                    
                # Create Sonifier
                sonifier = Sonifier(system = 'stereo', duration=duration)
                    
                # Generate sonification
                soni = sonifier.generate(
                    data_1, data_2, prop_1, prop_2
                )
                    
                # Render and save audio
                soni.render()
                audio_file = "eclipse_sonification.wav"
                soni.save_stereo(audio_file)
                print(f"✓ Generated sonification")
                print()
                
                make_video()
                # Display audio player
                display(Video("eclipse_sonification.mp4", width=500, height=400))
                    
            except Exception as e:
                print(f"✗ Error: {str(e)}")


    def display(self):
        """Display video on left, controls and audio on right with flexible sizing."""

        # Left side: Video display
        video_box = Box([self.video_area], layout=Layout(
            flex='1 1 auto',
            overflow='auto',
            min_width='500px'
        ))
        
        # Right side: Controls and output stacked vertically
        right_panel = VBox([self.selector_panel, self.output_area], layout=Layout(
            flex='0 0 auto',
            overflow='visible'
        ))
        
        controls_box = Box([right_panel], layout=Layout(
            flex='0 0 auto',
            overflow='visible',
            min_width='300px'
        ))

        # Side-by-side layout with video on left, controls on right
        display(HBox([video_box, controls_box], layout=Layout(
            display='flex',
            flex_flow='row',
            align_items='flex-start',
            width='100%',
            height='auto',
            gap='10px'
        )))


def find_widget_by_tag(container, tag):
    """
    Recursively search through a container for a widget with a specific custom tag.
    
    Parameters:
    -----------
    container : widget
        A widget container (e.g., VBox, HBox, Box).
    tag : str
        The custom tag to search for.
    
    Returns:
    --------
    widget or None
        The widget if found, otherwise None.
    """
    # Check if the container itself has the tag
    if hasattr(container, 'tag') and container.tag == tag:
        return container

    # If the container has children, search recursively
    if hasattr(container, 'children'):
        for child in container.children:
            found_widget = find_widget_by_tag(child, tag)
            if found_widget:
                return found_widget
    
    return None

In [ ]:
# Initialize the sonification UI
# The UI will automatically fetch data, organize by month, and sonify when you click Generate
app = SonificationUI()
app.display()